In [ ]:
import pandas as pd

ANTRO_PATH = "antropometria-dataset2.csv"
ADULT_PATH = "adult (1).csv"

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)

In [ ]:
antro = pd.read_csv(ANTRO_PATH)
adult = pd.read_csv(ADULT_PATH, skipinitialspace=True)

adult = adult[~adult["marital-status"].str.startswith("test_category")]

print(f"antropometria rows: {len(antro)} | adult rows (clean): {len(adult)}\n")

antropometria rows: 18640 | adult rows (clean): 32561



In [ ]:
antro = pd.read_csv(ANTRO_PATH)
adult = pd.read_csv(ADULT_PATH, skipinitialspace=True)

adult = adult[~adult["marital-status"].str.startswith("test_category")]

print(f"antropometria rows: {len(antro)} | adult rows (clean): {len(adult)}\n")

antropometria rows: 18640 | adult rows (clean): 32561



In [ ]:
q1 = antro.pivot_table(index="desc_ent", values="peso", aggfunc="mean", margins=True,
                       margins_name="TOTAL").round(2)
q1.columns = ["avg_weight_kg"]
print("Q1. Average weight (kg) by state")
print(q1, "\n")

morelos = antro.loc[antro["desc_ent"] == "MORELOS", "peso"].mean()
print(f"ANSWER Q1 -> Average weight of people from MORELOS: {morelos:.2f} kg\n")

Q1. Average weight (kg) by state
                                 avg_weight_kg
desc_ent                                      
AGUASCALIENTES                           66.36
BAJA CALIFORNIA                          68.47
BAJA CALIFORNIA SUR                      71.81
CAMPECHE                                 65.42
CHIAPAS                                  57.61
CHIHUAHUA                                66.66
COAHUILA DE ZARAGOZA                     68.15
COLIMA                                   68.74
DISTRITO FEDERAL                         65.12
DURANGO                                  66.35
GUANAJUATO                               63.94
GUERRERO                                 60.47
HIDALGO                                  60.00
JALISCO                                  66.70
MEXICO                                   60.68
MICHOACAN DE OCAMPO                      64.90
MORELOS                                  63.54
NAYARIT                                  66.54
NUEVO LEON                 

In [ ]:
by_sex = antro.pivot_table(index="desc_ent", columns="sexo", values="peso",
                           aggfunc="mean", margins=True, margins_name="TOTAL").round(2)
by_sex.columns = ["avg_weight_sex_1", "avg_weight_sex_2", "avg_weight_all"]
q2 = q1.join(by_sex[["avg_weight_sex_1", "avg_weight_sex_2"]])
print("Q2. Average weight by state, overall and by gender")
print(q2, "\n")

Q2. Average weight by state, overall and by gender
                                 avg_weight_kg  avg_weight_sex_1  avg_weight_sex_2
desc_ent                                                                          
AGUASCALIENTES                           66.36             69.99             63.25
BAJA CALIFORNIA                          68.47             71.39             65.89
BAJA CALIFORNIA SUR                      71.81             75.56             68.17
CAMPECHE                                 65.42             68.62             62.92
CHIAPAS                                  57.61             58.37             57.10
CHIHUAHUA                                66.66             68.89             64.82
COAHUILA DE ZARAGOZA                     68.15             68.87             67.47
COLIMA                                   68.74             71.00             66.98
DISTRITO FEDERAL                         65.12             69.46             61.65
DURANGO                             

In [ ]:
q3 = antro.pivot_table(index="desc_ent", columns="sexo", values="peso",
                       aggfunc=["mean", "count"]).round(2)
q3.columns = [f"{stat}_sex_{sex}" for stat, sex in q3.columns]
q3["diff_sex2_minus_sex1"] = (q3["mean_sex_2"] - q3["mean_sex_1"]).round(2)
print("Q3. Average weight (kg) of sex 1 and sex 2 by state")
print(q3, "\n")
print("Overall mean by sex:\n", antro.groupby("sexo")["peso"].mean().round(2), "\n")
print("States where sex 2 is heavier than sex 1:", int((q3["diff_sex2_minus_sex1"] > 0).sum()), "of", len(q3))
print("Mean sex gap across states:", round(q3["diff_sex2_minus_sex1"].mean(), 2), "kg\n")

Q3. Average weight (kg) of sex 1 and sex 2 by state
                                 mean_sex_1  mean_sex_2  count_sex_1  count_sex_2  diff_sex2_minus_sex1
desc_ent                                                                                               
AGUASCALIENTES                        69.99       63.25          302          353                 -6.74
BAJA CALIFORNIA                       71.39       65.89          264          300                 -5.50
BAJA CALIFORNIA SUR                   75.56       68.17          261          269                 -7.39
CAMPECHE                              68.62       62.92          237          304                 -5.70
CHIAPAS                               58.37       57.10          248          360                 -1.27
CHIHUAHUA                             68.89       64.82          265          322                 -4.07
COAHUILA DE ZARAGOZA                  68.87       67.47          299          313                 -1.40
COLIMA      

In [ ]:
q5 = adult.pivot_table(index="marital-status", values="age", aggfunc="count",
                       margins=True, margins_name="TOTAL")
q5.columns = ["count"]
q5 = q5.sort_values("count", ascending=False)
q5 = pd.concat([q5.drop("TOTAL"), q5.loc[["TOTAL"]]])
print("Q5. People by marital status")
print(q5, "\n")

Q5. People by marital status
                       count
marital-status              
Married-civ-spouse     14976
Never-married          10683
Divorced                4443
Separated               1025
Widowed                  993
Married-spouse-absent    418
Married-AF-spouse         23
TOTAL                  32561 



In [ ]:
q6 = q5.copy()
q6["percent"] = (q6["count"] / adult.shape[0] * 100).round(2)
print("Q6. People by marital status with percentage")
print(q6)

Q6. People by marital status with percentage
                       count  percent
marital-status                       
Married-civ-spouse     14976    45.99
Never-married          10683    32.81
Divorced                4443    13.65
Separated               1025     3.15
Widowed                  993     3.05
Married-spouse-absent    418     1.28
Married-AF-spouse         23     0.07
TOTAL                  32561   100.00


In [ ]:
for name, table in {"q1": q1, "q2": q2, "q3": q3, "q5_q6": q6}.items():
    table.to_csv(f"{name}_result.csv")

In [ ]:
med = antro.pivot_table(index="desc_ent", columns="sexo", values="peso",
                        aggfunc="median", margins=True)
print(med)
print(med.loc["YUCATAN"])
print(med.loc["YUCATAN", 2])

sexo                                   1        2      All
desc_ent                                                  
AGUASCALIENTES                   69.9500  63.7000  66.6000
BAJA CALIFORNIA                  70.6500  64.2000  67.9500
BAJA CALIFORNIA SUR              76.6500  68.5500  71.2500
CAMPECHE                         67.3500  62.3750  64.3500
CHIAPAS                          59.2000  56.0250  57.2000
CHIHUAHUA                        69.3000  64.6250  66.3000
COAHUILA DE ZARAGOZA             68.3000  67.3000  67.7000
COLIMA                           71.7000  66.0000  68.9000
DISTRITO FEDERAL                 69.0000  61.4500  64.0250
DURANGO                          70.3500  62.5000  65.7000
GUANAJUATO                       65.0625  60.3250  62.0625
GUERRERO                         60.5000  57.2000  57.9500
HIDALGO                          63.3000  57.9000  59.7000
JALISCO                          65.5875  63.3500  64.2000
MEXICO                           62.4500  56.9500  59.55